# Phase 4: fine-tune the text model on a free Colab GPU

This notebook trains the multilingual DistilBERT price model from `src/text_model.py` on a Colab GPU and gives you back **one zip file** to unpack in your project.

**Before you start:** in the menu, choose **Runtime → Change runtime type → T4 GPU → Save**.

**You need:**
1. Phase 4 pushed to GitHub on the branch `phase-4`. The notebook downloads your code from there.
2. Your processed data file from your computer: `S:\AirBNB\airbnb-price-estimator\data\processed\listings.parquet` (about 50 MB).

Run the cells from top to bottom with **Shift + Enter**. Training takes about 30–40 minutes.

## 1. Check the GPU
You should see a table naming a **Tesla T4**. If you see an error instead, change the runtime type as described above.

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## 2. Download your code from GitHub

In [2]:
REPO_URL = "https://github.com/Sammers45/airbnb-price-estimator.git"
BRANCH = "phase-4"


%cd /content
!rm -rf airbnb-price-estimator
!git clone --branch {BRANCH} --depth 1 {REPO_URL}
%cd /content/airbnb-price-estimator
!git log -1 --oneline

/content
Cloning into 'airbnb-price-estimator'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 25 (delta 0), reused 23 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 64.83 KiB | 1.38 MiB/s, done.
/content/airbnb-price-estimator
2e882d9 (grafted, HEAD -> phase-4, origin/phase-4) chore: ignore local MLFlow database


## 3. Install packages
Colab already has PyTorch and Transformers. This adds the few packages the project uses on top.

In [3]:
!pip install -q "polars>=1.5" pyyaml
import polars
import torch
import transformers

print(
    "torch",
    torch.__version__,
    "| transformers",
    transformers.__version__,
    "| polars",
    polars.__version__,
)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "NONE - change the runtime type!",
)

torch 2.11.0+cpu | transformers 5.22.0 | polars 1.35.2
GPU: NONE - change the runtime type!


## 4. Upload your processed data

Click **Choose Files** and select `data\processed\listings.parquet` from your project folder. Use your file, not a rebuilt one, so the folds match the baseline exactly.

*Alternative:* if the file is in your Google Drive, use the commented lines in the next cell instead.

In [ ]:
import pathlib
import shutil

import polars as pl
from google.colab import files

target = pathlib.Path("data/processed/listings.parquet")
target.parent.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # pick listings.parquet
name = next(iter(uploaded))
shutil.move(name, target)

# --- Google Drive alternative ---
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copy("/content/drive/MyDrive/airbnb/listings.parquet", target)

df = pl.read_parquet(target)
print(df.shape, "| folds:", sorted(df["fold"].unique().to_list()))
assert df.height == 5555 and "text" in df.columns and "fold" in df.columns, (
    "Unexpected file - upload data/processed/listings.parquet"
)

## 5. Run the tests (about 30 seconds)
This checks the code on a tiny model before spending GPU time. It should end with **passed**.

In [ ]:
!python -m pytest -q -p no:cacheprovider tests/test_text_model.py

## 6. Train (about 30–40 minutes)

This fine-tunes one model per fold (5 folds, up to 3 epochs each, with the best epoch picked on a slice of the training data), then one final model on all rows for the web app. The log shows each fold's MdAPE as it finishes.

If Colab disconnects, run cells 2–6 again.

In [ ]:
!python -m src.text_model train --no-mlflow

## 7. Results

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("reports/text_model.md", encoding="utf-8").read()))

## 8. Download the results

This creates `phase4_results.zip` (about 300 MB, mostly the final model). If the browser download is slow or fails, use the Google Drive option in the next cell.

In [ ]:
import pathlib
import zipfile

keep = [
    "data/predictions/text_oof.parquet",
    "reports/text_model.md",
    "reports/text_model_metrics.json",
    *[str(p) for p in pathlib.Path("models/text_model").glob("*")],
]
with zipfile.ZipFile("phase4_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p)
        print("added", p)
files.download("phase4_results.zip")

In [ ]:
# Optional: save the zip to Google Drive instead (more reliable for large files)
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copy("phase4_results.zip", "/content/drive/MyDrive/phase4_results.zip")

## 9. Back on your computer

1. Move `phase4_results.zip` into `S:\AirBNB\airbnb-price-estimator` and extract it **there**. The files land in `data\predictions`, `reports` and `models\text_model`.
2. Record the run in your local MLflow:
   ```
   python -m src.text_model log
   ```
3. Send `reports\text_model.md` for review.